In [22]:
import random
import math
from collections import Counter

file_path = "../assignment1/word_tokenized.txt"
K = 0.3
MAX_SENTENCES = 250000
random.seed(42)

def read_sentences():
    sentence = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            word = line.strip()
            if word:
                sentence.append(word)
            elif sentence:
                yield sentence
                sentence = []
        if sentence:
            yield sentence

sentences = []
for sentence in read_sentences():
    sentences.append(sentence)
    if len(sentences) >= MAX_SENTENCES:
        break

random.shuffle(sentences)
dev_set = sentences[:1000]
test_set = sentences[1000:2000]
train_set = sentences[2000:]

print(f"Training set size: {len(train_set)} sentences")
print(f"Development set size: {len(dev_set)} sentences")
print(f"Test set size: {len(test_set)} sentences")

Training set size: 248000 sentences
Development set size: 1000 sentences
Test set size: 1000 sentences


In [23]:
word_counts = Counter(word for sentence in train_set for word in sentence)

MIN_COUNT = 2

vocab = {word for word, count in word_counts.items() if count >= MIN_COUNT}
vocab.add("<UNK>")

def replace_oov(data):
    return [[word if word in vocab else "<UNK>" for word in sentence]
            for sentence in data]

train_set = replace_oov(train_set)
dev_set = replace_oov(dev_set)
test_set = replace_oov(test_set)

V = len(vocab)
N = sum(len(sentence) for sentence in train_set)

print("Training tokens:", N)
print("Vocabulary size:", V)

Training tokens: 4637205
Vocabulary size: 62410


In [24]:
def build_ngram_counts(n):
    counts = Counter()
    contexts = Counter()

    for sentence in train_set:
        for i in range(len(sentence) - n + 1):
            ngram = tuple(sentence[i:i+n])
            counts[ngram] += 1
            if n > 1:
                contexts[ngram[:-1]] += 1

    return counts, contexts

In [ ]:
unigram_counts, _ = build_ngram_counts(1)

def unigram_perplexity(dataset, k):
    denominator = N + k * V
    log_prob = 0
    total = 0
    for sentence in dataset:
        for word in sentence:
            log_prob += math.log((unigram_counts[(word,)] + k) / denominator)  # wrap word in a tuple
            total += 1
    return math.exp(-log_prob / total)

sample = next(iter(unigram_counts))
c = unigram_counts[sample]

print("Unigram")
print(f"Unique unigrams: {len(unigram_counts)}")
print(f"Add-1 sample probability: {(c + 1) / (N + V):.8f}")
print(f"Add-1 Dev Perplexity: {unigram_perplexity(dev_set, 1):.4f}")
print(f"Add-1 Test Perplexity: {unigram_perplexity(test_set, 1):.4f}")
print(f"Add-K (K={K}) sample probability: {(c + K) / (N + K * V):.8f}")
print(f"Add-K Dev Perplexity: {unigram_perplexity(dev_set, K):.4f}")
print(f"Add-K Test Perplexity: {unigram_perplexity(test_set, K):.4f}")

del unigram_counts

Unigram
Unique unigrams: 62410
Add-1 sample probability: 0.00127287
Add-1 Dev Perplexity: 4699615.0000
Add-1 Test Perplexity: 4699615.0000
Add-K (K=0.3) sample probability: 0.00128466
Add-K Dev Perplexity: 15519760.0000
Add-K Test Perplexity: 15519760.0000


In [26]:
bigram_counts, bigram_contexts = build_ngram_counts(2)

def bigram_perplexity(dataset, k):
    log_prob = 0
    total = 0
    for sentence in dataset:
        for i in range(1, len(sentence)):
            ngram = (sentence[i-1], sentence[i])
            context = (sentence[i-1],)
            log_prob += math.log((bigram_counts[ngram] + k) / (bigram_contexts[context] + k * V))
            total += 1
    return math.exp(-log_prob / total)

sample = next(iter(bigram_counts))
c = bigram_counts[sample]
cc = bigram_contexts[sample[:-1]]

print("Bigram")
print(f"Unique bigrams: {len(bigram_counts)}")
print(f"Add-1 sample probability: {(c + 1) / (cc + V):.8f}")
print(f"Add-1 Dev Perplexity: {bigram_perplexity(dev_set, 1):.4f}")
print(f"Add-1 Test Perplexity: {bigram_perplexity(test_set, 1):.4f}")
print(f"Add-K (K={K}) sample probability: {(c + K) / (cc + K * V):.8f}")
print(f"Add-K Dev Perplexity: {bigram_perplexity(dev_set, K):.4f}")
print(f"Add-K Test Perplexity: {bigram_perplexity(test_set, K):.4f}")

del bigram_counts, bigram_contexts

Bigram
Unique bigrams: 1124758
Add-1 sample probability: 0.00039489
Add-1 Dev Perplexity: 2903.6041
Add-1 Test Perplexity: 2833.1618
Add-K (K=0.3) sample probability: 0.00106534
Add-K Dev Perplexity: 1532.4929
Add-K Test Perplexity: 1496.3625


In [27]:
trigram_counts, trigram_contexts = build_ngram_counts(3)

def trigram_perplexity(dataset, k):
    log_prob = 0
    total = 0
    for sentence in dataset:
        for i in range(2, len(sentence)):
            ngram = tuple(sentence[i-2:i+1])
            context = ngram[:-1]
            log_prob += math.log((trigram_counts[ngram] + k) / (trigram_contexts[context] + k * V))
            total += 1
    return math.exp(-log_prob / total)

sample = next(iter(trigram_counts))
c = trigram_counts[sample]
cc = trigram_contexts[sample[:-1]]

print("Trigram")
print(f"Unique trigrams: {len(trigram_counts)}")
print(f"Add-1 sample probability: {(c + 1) / (cc + V):.8f}")
print(f"Add-1 Dev Perplexity: {trigram_perplexity(dev_set, 1):.4f}")
print(f"Add-1 Test Perplexity: {trigram_perplexity(test_set, 1):.4f}")
print(f"Add-K (K={K}) sample probability: {(c + K) / (cc + K * V):.8f}")
print(f"Add-K Dev Perplexity: {trigram_perplexity(dev_set, K):.4f}")
print(f"Add-K Test Perplexity: {trigram_perplexity(test_set, K):.4f}")

del trigram_counts, trigram_contexts

Trigram
Unique trigrams: 2680078
Add-1 sample probability: 0.00003203
Add-1 Dev Perplexity: 21045.0348
Add-1 Test Perplexity: 20946.5427
Add-K (K=0.3) sample probability: 0.00006934
Add-K Dev Perplexity: 13368.6576
Add-K Test Perplexity: 13333.7206


In [28]:
quadrigram_counts, quadrigram_contexts = build_ngram_counts(4)

def quadrigram_perplexity(dataset, k):
    log_prob = 0
    total = 0
    for sentence in dataset:
        for i in range(3, len(sentence)):
            ngram = tuple(sentence[i-3:i+1])
            context = ngram[:-1]
            log_prob += math.log((quadrigram_counts[ngram] + k) / (quadrigram_contexts[context] + k * V))
            total += 1
    return math.exp(-log_prob / total)

sample = next(iter(quadrigram_counts))
c = quadrigram_counts[sample]
cc = quadrigram_contexts[sample[:-1]]

print("Quadrigram")
print(f"Unique quadrigrams: {len(quadrigram_counts)}")
print(f"Add-1 sample probability: {(c + 1) / (cc + V):.8f}")
print(f"Add-1 Dev Perplexity: {quadrigram_perplexity(dev_set, 1):.4f}")
print(f"Add-1 Test Perplexity: {quadrigram_perplexity(test_set, 1):.4f}")
print(f"Add-K (K={K}) sample probability: {(c + K) / (cc + K * V):.8f}")
print(f"Add-K Dev Perplexity: {quadrigram_perplexity(dev_set, K):.4f}")
print(f"Add-K Test Perplexity: {quadrigram_perplexity(test_set, K):.4f}")

del quadrigram_counts, quadrigram_contexts

Quadrigram
Unique quadrigrams: 3397923
Add-1 sample probability: 0.00003205
Add-1 Dev Perplexity: 45368.5181
Add-1 Test Perplexity: 45660.6823
Add-K (K=0.3) sample probability: 0.00006943
Add-K Dev Perplexity: 37811.7069
Add-K Test Perplexity: 38173.0421
